# 🔍 Step 3 of RAG — Vector Database Retrieval

Now we test the three embedding approaches stored in our PostgreSQL vector database:

- **Dense retrieval** — Single 1024-dim vector per chunk, fast cosine similarity
- **Sparse retrieval** — Lexical token weights, keyword-based matching  
- **ColBERT retrieval** — Multi-vector per chunk, highest precision

We'll query all three with the same question and compare results.

## 0. Setup & Imports

In [ ]:
import os
import csv
import json
import random
import textwrap
import numpy as np
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch
from FlagEmbedding import BGEM3FlagModel

### Load config & connect to database

In [ ]:
# Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

# Load secrets
load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env file")

# Database connection
DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)

def get_connection():
    """Return a new psycopg2 connection."""
    return psycopg2.connect(DATABASE_URL)

# SQL directory
sql_dir = (Path.cwd() / "sql").resolve()

print("✅ Config loaded and database connection ready")

### Load embedding model

In [ ]:
print(f"Loading {config.embedding.model}...")
model = BGEM3FlagModel(config.embedding.model, use_fp16=True)
print("✅ Model loaded")

---

## 1. Prepare Query

Load a test question and embed it with all three approaches.

In [ ]:
# Load test questions from CSV
csv_path = (Path.cwd() / "../assets/benchmark_datasets/alice_test_set.csv").resolve()
with open(csv_path, "r", encoding="utf-8") as f:
    test_set = list(csv.DictReader(f))

# Pick a random question
sample = random.choice(test_set)
QUERY = sample["question"]
EXPECTED = sample["answer"]

print(f"Query   : {QUERY}")
print(f"Expected: {EXPECTED}")

# Embed the query with all approaches
q_out = model.encode(
    [QUERY],
    max_length=config.embedding.max_length,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

q_dense = q_out["dense_vecs"][0]
q_sparse = q_out["lexical_weights"][0]
q_colbert = q_out["colbert_vecs"][0]

print(f"\nDense   : {q_dense.shape}")
print(f"Sparse  : {len(q_sparse)} non-zero tokens")
print(f"ColBERT : {q_colbert.shape}")

TOP_K = config.retrieval.top_k  # Number of results to retrieve

---

## 2. Dense Retrieval

Fast cosine similarity search using the HNSW index.

In [ ]:
def retrieve_dense(q_dense: np.ndarray, top_k: int) -> list[dict]:
    """Dense retrieval using cosine similarity."""
    sql = (sql_dir / "09_query_dense.sql").read_text()
    vec_str = f"[{','.join(map(str, q_dense.tolist()))}]"
    
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(sql, (vec_str, vec_str, top_k))
        results = cur.fetchall()
        
        return [
            {
                "chunk_id": row[0],
                "chunk_text": row[1],
                "score": float(row[2])
            }
            for row in results
        ]
    finally:
        cur.close()
        conn.close()


dense_results = retrieve_dense(q_dense, TOP_K)

print(f"🔵 Dense — top {TOP_K} results\n")
for rank, r in enumerate(dense_results, 1):
    print(f"  Rank {rank} | score={r['score']:.4f} | chunk_id={r['chunk_id']}")
    print(f"  {textwrap.fill(r['chunk_text'][:200], width=70, subsequent_indent='  ')}\n")

---

## 3. Sparse Retrieval

Lexical matching based on token overlap and weights.

In [ ]:
def retrieve_sparse(q_sparse: dict, top_k: int) -> list[dict]:
    """Sparse retrieval using lexical token matching."""
    sql = (sql_dir / "10_query_sparse.sql").read_text()
    
    # Convert sparse dict to JSONB format
    sparse_json = json.dumps({str(k): float(v) for k, v in q_sparse.items()})
    
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(sql, (sparse_json, top_k))
        results = cur.fetchall()
        
        return [
            {
                "chunk_id": row[0],
                "chunk_text": row[1],
                "score": float(row[2])
            }
            for row in results
        ]
    finally:
        cur.close()
        conn.close()


sparse_results = retrieve_sparse(q_sparse, TOP_K)

print(f"🟡 Sparse — top {TOP_K} results\n")
for rank, r in enumerate(sparse_results, 1):
    print(f"  Rank {rank} | score={r['score']:.4f} | chunk_id={r['chunk_id']}")
    print(f"  {textwrap.fill(r['chunk_text'][:200], width=70, subsequent_indent='  ')}\n")

---

## 4. ColBERT Retrieval

Multi-vector retrieval using MaxSim scoring.

In [ ]:
def retrieve_colbert(q_colbert: np.ndarray, top_k: int) -> list[dict]:
    """ColBERT MaxSim retrieval using simple SQL approach."""
    # Load SQL from file
    sql = (sql_dir / "11_query_colbert.sql").read_text()
    
    candidate_pool = top_k * 3  # Get more candidates per token for better results
    chunk_scores = {}
    
    conn = get_connection()
    cur = conn.cursor()
    try:
        for q_token_vec in q_colbert:
            # Convert single query token to vector string
            vec_str = f"[{','.join(map(str, q_token_vec.tolist()))}]"
            cur.execute(sql, (vec_str, candidate_pool))
            
            for chunk_id, chunk_text, max_sim in cur.fetchall():
                if chunk_id not in chunk_scores:
                    chunk_scores[chunk_id] = {
                        "chunk_text": chunk_text,
                        "token_sims": []
                    }
                chunk_scores[chunk_id]["token_sims"].append(float(max_sim))
    finally:
        cur.close()
        conn.close()
    
    # Compute MaxSim score (average of max similarities per query token)
    results = []
    for chunk_id, data in chunk_scores.items():
        if data["token_sims"]:
            maxsim_score = float(np.mean(data["token_sims"]))
            results.append({
                "chunk_id": chunk_id,
                "chunk_text": data["chunk_text"], 
                "score": maxsim_score
            })
    
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


colbert_results = retrieve_colbert(q_colbert, TOP_K)

print(f"🟢 ColBERT — top {TOP_K} results\n")
for rank, r in enumerate(colbert_results, 1):
    print(f"  Rank {rank} | score={r['score']:.4f} | chunk_id={r['chunk_id']}")
    print(f"  {textwrap.fill(r['chunk_text'][:200], width=70, subsequent_indent='  ')}\n")

---

## 5. Results Comparison

Compare the three approaches side by side.

In [ ]:
print("=" * 80)
print(f"QUERY: {QUERY}")
print(f"EXPECTED: {EXPECTED}")
print("=" * 80)

print("\n📊 RESULTS COMPARISON\n")

for rank in range(1, TOP_K + 1):
    print(f"── Rank {rank} ──")
    
    if rank <= len(dense_results):
        r = dense_results[rank - 1]
        print(f"🔵 Dense   | score={r['score']:.4f} | chunk_id={r['chunk_id']} | {r['chunk_text'][:50]}...")
    
    if rank <= len(sparse_results):
        r = sparse_results[rank - 1]
        print(f"🟡 Sparse  | score={r['score']:.4f} | chunk_id={r['chunk_id']} | {r['chunk_text'][:50]}...")
    
    if rank <= len(colbert_results):
        r = colbert_results[rank - 1]
        print(f"🟢 ColBERT | score={r['score']:.4f} | chunk_id={r['chunk_id']} | {r['chunk_text'][:50]}...")
    
    print()

---

## 6. Summary

| Approach | Strengths | Weaknesses |
|---|---|---|
| **Dense** | Fast, good semantic understanding | May miss exact keywords |
| **Sparse** | Excellent keyword matching | Limited semantic understanding |
| **ColBERT** | Best of both worlds, highest precision | Slower, more storage |

### Key Takeaways

- **Dense retrieval** excels at semantic similarity but may miss exact term matches
- **Sparse retrieval** catches keyword-based queries that dense might miss
- **ColBERT** typically provides the best overall performance by combining fine-grained token matching with semantic understanding
- In production, you might use **hybrid approaches** that combine multiple methods for optimal results